In [206]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [207]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.preprocessing import LabelEncoder

In [208]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [209]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [210]:
encoder = LabelEncoder()

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,

In [211]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [212]:
x_ = torch.from_numpy(train_x.to_numpy()).type('torch.FloatTensor')
y_ = torch.from_numpy(train_y.to_numpy()).type('torch.FloatTensor')
sub = torch.from_numpy(test.to_numpy()).type('torch.FloatTensor')

In [213]:
train_x = torch.utils.data.DataLoader(x_,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y_,batch_size=batch_size)
test_sub = torch.utils.data.DataLoader(sub,batch_size=batch_size)

In [214]:
def weights_init_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        
class ENCODER(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(57, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),
            
            nn.Linear(1524, 2064),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(2064, 4064),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),
            
            nn.Linear(4064, 8064),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(8064, 16024),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),

            nn.Linear(16024, 30000)
            
        )

    def forward(self,x):
        
        return self.rafire(x).reshape(3,100,100)

encoder = ENCODER().float()
encoder= nn.DataParallel(encoder).to(device)
encoder.apply(weights_init_)
torch.save(encoder.state_dict(),f'/kaggle/working/encoder_weight.pth')

In [215]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2)
        )
    def forward(self, x):
        x = encoder(x)
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [216]:
EFF_NET = high_regressor().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
EFF_NET.apply(weights_init)

DataParallel(
  (module): high_regressor(
    (rafire): Sequential(
      (0): Linear(in_features=57, out_features=1524, bias=True)
      (1): BatchNorm1d(1524, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
      (3): Linear(in_features=1524, out_features=824, bias=True)
      (4): BatchNorm1d(824, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.01)
      (6): Linear(in_features=824, out_features=424, bias=True)
      (7): BatchNorm1d(424, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): LeakyReLU(negative_slope=0.01)
      (9): Linear(in_features=424, out_features=824, bias=True)
      (10): BatchNorm1d(824, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (11): LeakyReLU(negative_slope=0.01)
      (12): Linear(in_features=824, out_features=1524, bias=True)
      (13): BatchNorm1d(1524, eps=1e-05, momentum=0.1, affine=True, tr

In [217]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr,betas=(beta1, 0.999))
scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [ ]:
high_rf,i,j_r,k,z_w_=100000,0,0,0,[]
    
correct,total=0,0

while(True):
    optimizer.zero_grad()
    for data,label in zip(train_x,train_y):
        label=torch.from_numpy(numpy.array([[0,1] if i==1 else [1,0] for i in label])).float().to(device)
        output = EFF_NET(data.to(device))
        err_real = criterion(output, label.to(device))
        err_real.backward()
        optimizer.step()
        #print(err_real.item())
    
    print(f"EPOCH : {i} LOSS_wl : {err_real.item()}")

    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_,j_r=[],0
            print(f"lr_br:{optimizer.param_groups[0]['lr']}".upper())
            scheduler.step()
            print(f"lr_up:{optimizer.param_groups[0]['lr']}".upper())
            
    z_w_.append(err_real.item())
    if err_real.item()<high_rf:
            high_rf=err_real.item()
            torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()}.pth')
    if i==20:
        break
    i+=1

EPOCH : 0 LOSS_wl : 0.7879499793052673
EPOCH : 1 LOSS_wl : 0.6756342649459839
EPOCH : 2 LOSS_wl : 0.7018910050392151
EPOCH : 3 LOSS_wl : 0.6804371476173401
EPOCH : 4 LOSS_wl : 0.6818597912788391
EPOCH : 5 LOSS_wl : 0.6458982825279236
LR_BR:0.0001
LR_UP:8.6E-05
EPOCH : 6 LOSS_wl : 0.6605608463287354
EPOCH : 7 LOSS_wl : 0.6529858112335205
EPOCH : 8 LOSS_wl : 0.6402683258056641
EPOCH : 9 LOSS_wl : 0.6543508768081665
EPOCH : 10 LOSS_wl : 0.6654679775238037
EPOCH : 11 LOSS_wl : 0.650161862373352
EPOCH : 12 LOSS_wl : 0.6494987607002258
EPOCH : 13 LOSS_wl : 0.6515599489212036
EPOCH : 14 LOSS_wl : 0.6383691430091858
EPOCH : 15 LOSS_wl : 0.6562458872795105
EPOCH : 16 LOSS_wl : 0.6614735126495361
EPOCH : 17 LOSS_wl : 0.6639333963394165
